# The Five Patterns You'll Actually Use

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 5 §5.2–5.4 · type: concept-lab*  🔧 *This is the chapter's Build.*

You'll wire the exact provider layer the book builds — a `LLMProvider` **Protocol** (the
seam), an **Adapter** per vendor, a **Factory** driven by config, **Dependency Injection**
into an `Agent`, **Strategy** for retrieval, and a **Repository** hiding persistence — then
put a real domain rule (`Thread.append` enforcing a `token_budget`) where the data that owns
it lives. Everything runs offline through a `FakeProvider`; this is the skeleton the capstone
and every later part are fleshed out from.

## 🧠 Why this matters

Classic design patterns are a *vocabulary*, and in Python — with first-class functions and
structural typing — most of them collapse into a few lines. The trap is treating them as
cleverness. They aren't.

The one idea under all five patterns here: **each one creates a seam — a typed boundary where
one side can change, be faked, or be swapped without the other side knowing.** Patterns are
**pre-paid flexibility at the exact places experience says you'll need it.** In agentic systems
those places are stable and few: the model provider, the storage layer, the retrieval pipeline,
the tool registry.

Build the seam *once*, here, and you get the property that makes every later chapter possible:
testability. The `FakeProvider` in this notebook is precisely what makes Ch 7's `FakeLLM`
testing work — testability is a design property you earn at construction time, not bolt on
later.

## Objectives & prerequisites

**By the end you can:**

- Define a **Protocol** seam (`LLMProvider`) and write an **Adapter** (`FakeProvider`,
  `AnthropicProvider`) that conforms to it — no inheritance required.
- Concentrate the "which concrete class?" decision in a **Factory** (`make_provider`) driven
  by config, using `match`.
- **Inject** the provider into an `Agent` via its constructor — DI with no framework.
- Use **Strategy** (a Protocol *or* a plain function) and a **Repository** interface in domain
  terms.
- Put a business rule **in the domain object** (`Thread.append` raising `BudgetExceeded`) so
  it's true everywhere.

**Prereqs:** `05-01` (refactoring, seams); Ch 4 Protocols.

**Run first:** the Setup cell. Defaults to `MOCK=1`, so the whole notebook runs **free and
offline** via `FakeProvider`. A real Anthropic adapter is shown and only exercised if a key
is present and you set `COMPANION_MOCK=0`.

In [ ]:
# --- Setup -------------------------------------------------------------------
import os
import random
from typing import Protocol, runtime_checkable

from dotenv import load_dotenv

load_dotenv()  # secrets come from the environment only; never hardcode keys

# MOCK=1 (default): the factory builds a FakeProvider -> fully offline, deterministic,
# nothing billed. MOCK=0: the factory can build a real AnthropicProvider, which needs
# ANTHROPIC_API_KEY and bills a few short calls.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(5)  # determinism habit (the FakeProvider is already deterministic)

MODEL = "claude-opus-4-8"  # the book's default model

if MOCK:
    print("MOCK mode: provider layer runs on FakeProvider. No API key needed, nothing billed.")
else:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
            "Set it in your environment/.env, or use COMPANION_MOCK=1 for the offline path."
        )
    print(f"LIVE mode: the factory may build a real AnthropicProvider calling {MODEL}.")

## 1 · 🔧 The seam: an `LLMProvider` Protocol

A **Protocol** is a *structural* interface (Ch 4): any class with the right methods satisfies
it — no base class, no registration. That makes it the perfect seam, because your domain code
depends on the *shape* `complete(...)`, not on any concrete vendor.

`@runtime_checkable` lets us `isinstance`-check the shape in a demo cell (you wouldn't litter
production code with these — the type checker is the real enforcement).

In [ ]:
@runtime_checkable
class LLMProvider(Protocol):                       # the seam (DI target)
    async def complete(self, prompt: str, *, max_tokens: int) -> str: ...


print("LLMProvider Protocol defined. It is a shape, not a base class.")

## 2 · 🔧 Adapter: wrap each vendor SDK into *our* shape

An **Adapter** translates a vendor's API into the one shape the rest of the system sees. When
a vendor changes its API — they do — exactly **one file** changes.

We write two adapters:

- `FakeProvider` — offline, deterministic canned text. This is the seam that keeps the whole
  notebook (and CI, and readers without a key) free.
- `AnthropicProvider` — the book's real adapter (§5.3). We *define* it but only *instantiate*
  it on the live path, so importing `anthropic` is never required in MOCK mode.

In [ ]:
class FakeProvider:                                 # ADAPTER (offline): canned text
    """Deterministic stand-in for any LLM. Same shape as the real adapter."""
    def __init__(self, reply: str = "Mocked reply: I read your prompt."):
        self._reply = reply

    async def complete(self, prompt: str, *, max_tokens: int) -> str:
        # Deterministic + cheap: echo a tag so you can see the prompt reached us.
        return f"{self._reply} (saw {len(prompt)} chars, cap {max_tokens})"


class AnthropicProvider:                            # ADAPTER: vendor SDK -> our shape
    """The book's real adapter (§5.3). Only instantiated on the live path."""
    def __init__(self, api_key: str, model: str):
        import anthropic  # imported lazily so MOCK mode never needs the SDK
        self._client = anthropic.AsyncAnthropic(api_key=api_key)
        self._model = model

    async def complete(self, prompt: str, *, max_tokens: int) -> str:
        msg = await self._client.messages.create(
            model=self._model, max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return msg.content[0].text


# Both conform to the seam by SHAPE -- no inheritance:
print("FakeProvider satisfies LLMProvider:", isinstance(FakeProvider(), LLMProvider))
print("AnthropicProvider satisfies LLMProvider:",
      issubclass(AnthropicProvider, LLMProvider) or hasattr(AnthropicProvider, "complete"))

## 3 · 🔧 Factory: one place decides *which* concrete class

The **Factory** concentrates the "which vendor?" decision in a single function driven by
config. Add a vendor → add a `case`; nothing else in the system changes. Notice this is
*Open/Closed* in practice (§5.2): you extend behavior by adding a branch, not by editing
callers.

We use a tiny `Settings` object (Ch 4's `core/Settings`) so the factory reads config, never
the environment directly.

In [ ]:
from dataclasses import dataclass


@dataclass
class Settings:
    """Stand-in for Ch 4's core/Settings (only the fields this factory needs)."""
    llm_vendor: str = "fake"
    llm_model: str = MODEL
    llm_api_key: str = ""


def make_provider(settings: Settings) -> LLMProvider:   # FACTORY
    match settings.llm_vendor:
        case "fake":
            return FakeProvider()
        case "anthropic":
            return AnthropicProvider(settings.llm_api_key, settings.llm_model)
        case _:
            raise ValueError(f"unknown vendor: {settings.llm_vendor}")


# In MOCK mode we drive the offline branch; live mode flips one config field.
settings = Settings(
    llm_vendor="fake" if MOCK else "anthropic",
    llm_api_key=os.getenv("ANTHROPIC_API_KEY", ""),
)
provider = make_provider(settings)
print("factory built:", type(provider).__name__)

## 4 · 🔧 Dependency Injection: the `Agent` *receives* its provider

**Dependency injection** just means the `Agent` is *handed* its collaborators instead of
constructing them. No framework — it's a constructor argument. The payoff is testability (pass
a `FakeProvider`) and swappability (the factory decides the concrete type). FastAPI's `Depends`
(Ch 25) is this same idea wired into request handling.

### 🔮 Predict

We build the `Agent` with whatever `make_provider` returned, then run it. Below, predict
*before running the next cell*: if you change **only** `settings.llm_vendor` from `"fake"` to
`"anthropic"` (and provide a key), how many lines of `Agent` change? What downstream code has
to know the provider switched?

In [ ]:
import asyncio


class Agent:
    def __init__(self, llm: LLMProvider, persona: str = "You are a helpful agent."):
        self.llm = llm                              # DEPENDENCY INJECTION
        self.persona = persona

    async def reply(self, user_text: str) -> str:
        prompt = f"{self.persona}\nUser: {user_text}"
        return await self.llm.complete(prompt, max_tokens=256)


agent = Agent(provider)                              # inject the factory's product
out = asyncio.run(agent.reply("What is a seam?"))
print("Agent reply:", out)

**What you just saw.** The answer to the prediction: **zero** lines of `Agent` change, and
nothing downstream knows. Swapping the factory's branch swaps the *entire* model backend, and
`Agent`, `reply`, and every caller are untouched — because they depend on the `LLMProvider`
seam, not on a vendor. That is the whole point of the Protocol → Adapter → Factory → DI chain:
the volatile decision lives in one branch of one function.

## 5 · Strategy: behavior that varies behind a common interface

**Strategy** is the pattern you're looking at whenever behavior varies behind a common shape —
retrieval (keyword vs. vector vs. hybrid), chunking, routing. In Python a strategy can be a
Protocol implementation **or** just a function. We show both, because choosing the lighter one
is itself a senior call (don't reach for a class when a function will do).

In [ ]:
from typing import Callable

DOCS = ["alpha pricing guide", "beta refund policy", "gamma pricing tiers"]


# Strategy as a plain function (the lighter form):
def keyword_search(query: str, docs: list[str]) -> list[str]:
    return [d for d in docs if query.lower() in d.lower()]


def vector_search(query: str, docs: list[str]) -> list[str]:
    # Toy "vector" score: word overlap. Real version uses embeddings (Ch 13).
    q = set(query.lower().split())
    scored = sorted(docs, key=lambda d: len(q & set(d.lower().split())), reverse=True)
    return [d for d in scored if q & set(d.lower().split())]


Retriever = Callable[[str, list[str]], list[str]]   # the strategy's shape


def retrieve(query: str, strategy: Retriever) -> list[str]:
    return strategy(query, DOCS)                     # caller injects the strategy


print("keyword:", retrieve("pricing", keyword_search))
print("vector :", retrieve("pricing tiers", vector_search))

## 6 · Repository: hide persistence behind domain terms

A **Repository** exposes an interface in *domain* language — `get(thread_id)`, `save(thread)` —
so business logic never touches SQL or a vendor client. Here it's an in-memory dict; in Ch 30
it's Postgres. The domain code can't tell the difference, which is the point.

In [ ]:
class ThreadRepo(Protocol):                         # REPOSITORY seam (domain terms)
    def get(self, thread_id: str) -> "Thread": ...
    def save(self, thread: "Thread") -> None: ...


class InMemoryThreadRepo:                            # one offline implementation
    def __init__(self):
        self._store: dict = {}

    def get(self, thread_id: str):
        return self._store[thread_id]

    def save(self, thread) -> None:
        self._store[thread.id] = thread


print("ThreadRepo seam + InMemoryThreadRepo impl defined.")

## 7 · SOLID as a *change* heuristic, not commandments

Underneath all five patterns sit two words: **cohesion** (things that change together live
together) and **coupling** (things that change separately can be changed separately). The test
is never "is this class SOLID?" — it's: *pick a plausible change and count the files it
touches.* One or two: healthy. Seven across three packages: your boundaries are wrong, however
SOLID each class looks alone.

Below we make the heuristic concrete: "add a second LLM vendor" on the seamed design touches a
single file region (the factory's `match`), versus a tangled design that hardcodes the vendor
inside `Agent`.

In [ ]:
# Seamed design: "add vendor X" touches ONE place -- the factory branch.
seamed_change_sites = ["make_provider() -- add one `case`"]

# Tangled design (anti-pattern): Agent builds its own client inline, so the vendor
# choice is smeared across every class that talks to a model.
tangled_change_sites = [
    "Agent.__init__ -- swap the hardcoded client",
    "Summarizer.__init__ -- swap it again",
    "Classifier.__init__ -- and again",
    "every test -- now needs the new client + a key",
]

print(f"seamed:  {len(seamed_change_sites)} file/region to change")
for s in seamed_change_sites:
    print("   -", s)
print(f"tangled: {len(tangled_change_sites)} places to change")
for s in tangled_change_sites:
    print("   -", s)
print("\nSame feature. The seam made the change LOCAL -- that is what SOLID buys.")

## 8 · 🔧 Domain modeling: put the rule where the data lives

The single most consequential clean-code decision is **where the rules about your concepts
live**. The failure mode has a name — the **anemic domain model**: data classes with no
behavior, while the actual rules smear across handlers, "service" grab-bags, and workers,
drifting out of sync.

The cure (§5.4): put behavior *with* the data that owns it. `Thread.append` enforces its own
`token_budget` and raises `BudgetExceeded` — so "a thread enforces its budget" is true
**everywhere** (API, worker, test), because there's exactly one place the rule exists.

We use stdlib dataclasses here so the notebook needs no extra deps; the book uses Pydantic
`BaseModel`, but the design — *rule in the method* — is identical.

In [ ]:
from dataclasses import dataclass, field


class BudgetExceeded(Exception):
    """Raised when appending a message would exceed a thread's token budget."""
    def __init__(self, thread_id: str):
        super().__init__(f"thread {thread_id} exceeded its token budget")
        self.thread_id = thread_id


@dataclass
class Message:
    role: str
    content: str
    tokens: int


@dataclass
class Thread:
    id: str
    owner_id: str
    messages: list = field(default_factory=list)
    token_budget: int = 100  # small here so we can trip it on purpose

    def tokens_used(self) -> int:
        return sum(m.tokens for m in self.messages)

    def append(self, msg: Message) -> None:
        if self.tokens_used() + msg.tokens > self.token_budget:
            raise BudgetExceeded(self.id)            # the RULE lives HERE, once
        self.messages.append(msg)


thread = Thread(id="t1", owner_id="u1")
thread.append(Message("user", "hello", tokens=40))
thread.append(Message("assistant", "hi there", tokens=40))
print("tokens_used after two messages:", thread.tokens_used())

## 9 · ⚠️ Pitfall: the anemic model (and its mirror, over-abstraction)

Two failure modes bracket good design:

- **Anemic domain model** — the rule lives *outside* the data, copied into three handlers,
  drifting apart. Below we contrast it with the rich `Thread`: in the rich version, the budget
  check is impossible to forget because you *can't* append without going through it.
- **Over-abstraction** — interfaces with one implementation forever, `AbstractFactoryFactory`
  towers. That fails the same cognitive-load budget from `05-01`. Abstraction is a **loan**.

### 🔮 Predict

The next message would push us to 120 tokens against a budget of 100. Predict *before running*:
what exactly happens when we `thread.append` it — and which line of which file raises? (Contrast
that with the anemic version, where the check sits in a handler that a new code path can simply
forget to call.)

In [ ]:
# Rich domain object: the rule is unforgettable -- append() enforces it.
try:
    thread.append(Message("user", "a very long message", tokens=40))  # -> 120 > 100
    print("appended (you should NOT see this)")
except BudgetExceeded as exc:
    print(f"BudgetExceeded raised by Thread.append for {exc.thread_id!r} -- rule held.")


# Anemic anti-pattern, for contrast: data is dumb, the rule lives in a handler...
@dataclass
class AnemicThread:
    id: str
    messages: list = field(default_factory=list)
    token_budget: int = 100


def handler_append(t: AnemicThread, msg: Message) -> None:
    # ...and a SECOND code path can simply forget this check -> drift.
    if sum(m.tokens for m in t.messages) + msg.tokens > t.token_budget:
        raise BudgetExceeded(t.id)
    t.messages.append(msg)


anemic = AnemicThread(id="t2")
anemic.messages.append(Message("user", "snuck in", tokens=999))  # bypassed the rule!
print("anemic thread tokens (rule BYPASSED):", sum(m.tokens for m in anemic.messages))

**What you just saw.** With the rich `Thread`, the only way to add a message is `append()`,
so the budget rule is **enforced by construction** — `BudgetExceeded` fired exactly where the
data lives. With `AnemicThread`, a caller appended straight to `.messages` and sailed past the
limit: the rule, living in a handler, was trivially bypassed. Multiply that by every handler,
worker, and migration script, and you have rules that "exist" but aren't *true*.

## 10 · 📋 The clean-code & design checklist (self-audit)

Run this over the layer you just built — and over your next PR:

- [ ] **Names** state domain intent; scope matches descriptiveness; hard-to-name things get
  redesigned, not abbreviated.
- [ ] **Functions** do one job at one abstraction level; guard clauses over nesting; ≤3 args.
- [ ] **Change test:** a plausible requirement change touches one or two modules, not seven.
- [ ] **Abstraction loans are repaid:** every interface has a second implementation, a test
  seam, or an isolation reason — or it gets inlined.
- [ ] **Seams exist** at the volatile points — provider, storage, retrieval, tools — as
  Protocols with adapters, built by factories, injected via constructors.
- [ ] **Rules live in the domain layer**, once; handlers translate, infrastructure adapts;
  dependencies point inward.
- [ ] **Refactoring is continuous**, test-covered, small-stepped — never a big bang.
- [ ] **AI smells reviewed explicitly:** prompt spaghetti, dict-of-everything, swallowed
  exceptions, duplicated helpers.

## 🎯 Senior lens: abstraction is a loan

Every Protocol, adapter, and factory you add is a **loan** — it must be repaid by a *real*
second implementation, a *real* test seam, or a *real* isolation need. The junior move is to
abstract everything "to be safe"; the result is five-line classes scattered across twenty
files, which fails the same cognitive-load budget as the spaghetti it replaced.

Notice the loans in *this* notebook are all repaid: the `LLMProvider` seam already has **two**
implementations (`FakeProvider` *and* `AnthropicProvider`) and a **test** need (offline CI);
the `Retriever` strategy has two real strategies; the `ThreadRepo` will get a Postgres impl in
Ch 30. We did **not** abstract `Settings` behind an interface — it has one shape and no second
implementation in sight. Knowing which loans to take, and which to refuse, is the senior skill.
Take the loan where experience says volatility lives — provider, storage, retrieval, tools — and
nowhere else, *yet*.

## Recap

- **One idea, five patterns:** Protocol, Adapter, Factory, DI, Strategy, Repository each create
  a **seam** — a typed boundary one side can change/fake/swap without the other knowing.
- **The chain in action:** `LLMProvider` (seam) ← `FakeProvider`/`AnthropicProvider` (adapters)
  ← `make_provider` (factory) → injected into `Agent` (DI). Swapping the vendor changed **zero**
  lines of `Agent`.
- **SOLID is a change heuristic:** count the files a plausible change touches. Seams make the
  change *local*.
- **Rules live in the domain:** `Thread.append` enforces its `token_budget` and raises
  `BudgetExceeded`, so the rule is true everywhere — the cure for the **anemic model**.
- **Abstraction is a loan** repaid only by a second impl, a test seam, or an isolation need —
  knowing when *not* to abstract is the senior skill.

## Exercises

Use the empty cells below. (Solutions land in `solutions/` in Phase 2.)

1. **Add a vendor.** Add a `case "local"` to `make_provider` returning a `LocalProvider`
   adapter (canned text is fine). Predict *before running*: which existing code outside the
   factory has to change to use it? Confirm by building an `Agent` on the new vendor and calling
   `reply`.
2. **Repay (or refuse) a loan.** Add a `HybridRetriever` strategy that unions keyword + vector
   results, then argue in a comment whether the `Retriever` abstraction has now earned its keep.
   When would a *single* retrieval strategy make the Protocol an unpaid loan you should inline?
3. **Move a rule into the domain.** Give `Thread` a `close()` method and a `status` field that
   only allows `open → closed` (raise on any other transition). Predict the exact exception when
   you call `close()` twice, then show why putting this in `Thread` beats a `close_thread(t)`
   handler — using the anemic-bypass demo as your evidence.

In [ ]:
# Exercise 1 -- your code here

In [ ]:
# Exercise 2 -- your code here

In [ ]:
# Exercise 3 -- your code here

## Next

You built the provider/domain skeleton. From here the course flesh it out into a real backend.

- ▶️ **Where this goes next:** the testability you just earned (`FakeProvider`) is what makes
  Ch 7's safety harness and `FakeLLM` work — testability is a design property earned here, at
  construction time.
- 🎓 **Capstone:** this notebook *is* the seed of
  [`../../../capstone/`](../../../capstone/) — `providers/` (the `LLMProvider` Protocol +
  Anthropic adapter + `make_provider` factory) and `domain/` (`Thread`/`Message`/`Run` owning
  their rules, budget enforcement and status transitions). This chapter is the 🔧 Build that
  skeleton starts from.
- 🔧 **Blueprint:** the same Protocol → Adapter → Factory → DI shape recurs (authored there) in
  [`../../../blueprints/agent-loop/`](../../../blueprints/agent-loop/).
- 📖 **Book:** see §5.2 (SOLID without dogma), §5.3 (the patterns), §5.4 (domain modeling).